# DBSCAN reutilizável — pipeline genérico de clustering por densidade

Mesmo tratamento dado ao notebook de K-Means (parte 1): generalizar o notebook
original (Mall Customers, 2 features fixas, `eps`/`min_samples` escolhidos no
olho) para qualquer cenário — trocando um `DensityClusterConfig`, nenhuma
função abaixo muda.

**Isto não é uma cópia do padrão do K-Means.** DBSCAN é outro método, com outra
filosofia, e o pipeline reflete isso:

- **Não tem outlier-clipping antes do modelo.** No K-Means, outlier distorce o
  centróide e precisa ser tratado antes. No DBSCAN, ruído/outlier é o **output**
  do modelo, não um problema a esconder dele — se você limpar outlier antes,
  o DBSCAN não vai ter nada para marcar como ruído e a análise perde o sentido.
- **Não tem `k`.** Em troca, tem dois hiperparâmetros mais difíceis de
  escolher (`eps`, `min_samples`) — o pipeline automatiza a sugestão inicial
  (k-distance + detecção de joelho, heurística 2×D para `min_samples`), mas
  trate como ponto de partida, não veredito.
- **Não tem seed/`random_state` relevante para o resultado.** DBSCAN é
  determinístico dado `(eps, min_samples, metric)` — rodar de novo com os
  mesmos parâmetros dá o mesmo resultado (a menos de ambiguidade de fronteira
  em pontos de borda, um caso conhecido do algoritmo). Por isso a "estabilidade"
  aqui mede **sensibilidade a hiperparâmetro**, não a inicialização aleatória
  como no K-Means — são conceitos diferentes, tratados com funções diferentes.
- **Não tem `.predict()` nativo.** DBSCAN é transdutivo — o resultado depende
  do dataset de treino inteiro, não existe fronteira de decisão fixa para um
  ponto novo. `predict_new` aqui é uma aproximação documentada (atribuição ao
  core point mais próximo dentro de `eps`), não equivalente a re-treinar.

**Para reusar em outro cenário:** troque `data_path`, `numeric_features` e
`categorical_features` no `DensityClusterConfig` — igual ao notebook de
K-Means. Casos de uso típicos de DBSCAN vs. K-Means: detecção de anomalia/fraude
(ruído *é* o resultado que interessa), clusters de formato irregular
(geolocalização, trajetórias), ou quando você não sabe quantos grupos esperar.


In [ ]:
from __future__ import annotations

import pickle
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Definir o problema — `DensityClusterConfig`

`eps` e `min_samples` como `None` significam "sugira automaticamente" (seções
5 e 6). Defina os dois explicitamente quando já souber os valores certos para
o seu domínio.

In [ ]:
@dataclass
class DensityClusterConfig:
    '''Configuração de um cenário de clustering por densidade. Trocar cenário =
    trocar esta instância; nenhuma função abaixo precisa ser editada.'''
    data_path: str
    id_col: Optional[str] = None
    numeric_features: list = field(default_factory=list)
    categorical_features: list = field(default_factory=list)
    scale_method: str = "standard"      # "standard" | "robust"
    eps: Optional[float] = None         # None -> sugerido via k-distance + joelho
    min_samples: Optional[int] = None   # None -> sugerido via heuristica 2*D
    metric: str = "euclidean"
    random_state: int = 42              # usado so na projecao PCA de visualizacao
    model_path: str = "dbscan_pipeline.pkl"


# CONFIG -- reproduz o cenario do notebook original (Renda x Spending Score)
# CONFIG_CELL_MARKER
CONFIG = DensityClusterConfig(
    data_path="dataset/Mall_Customers.csv",
    id_col="CustomerID",
    numeric_features=["Annual Income (k$)", "Spending Score (1-100)"],
    categorical_features=[],
)

## 2. Perfilar os dados

Igual ao notebook de K-Means: checa tipo, % de nulo e cardinalidade das
colunas usadas. DBSCAN também não lida com `NaN`.

In [ ]:
def load_data(config: DensityClusterConfig) -> pd.DataFrame:
    df = pd.read_csv(config.data_path)
    used_cols = config.numeric_features + config.categorical_features
    missing = [c for c in used_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes no dataset: {missing}")
    return df


def profile_dataset(df: pd.DataFrame, config: DensityClusterConfig) -> pd.DataFrame:
    cols = config.numeric_features + config.categorical_features
    prof = pd.DataFrame({
        "dtype": df[cols].dtypes.astype(str),
        "pct_nulo": (df[cols].isna().mean() * 100).round(2),
        "n_unicos": df[cols].nunique(),
    })
    if prof["pct_nulo"].gt(0).any():
        warnings.warn("Ha colunas com nulo nas features do modelo -- trate antes de escalar.")
    return prof

## 3. Engenharia de features — scaling + encoding genéricos

Mesmo `ColumnTransformer` do notebook de K-Means: escala numéricas, one-hot
nas categóricas. **Sem etapa de outlier aqui** — ver nota na introdução.

Ressalva técnica: distância euclidiana sobre colunas one-hot misturadas com
numéricas padronizadas é uma aproximação, não uma métrica "correta" — cada
categoria one-hot contribui 0 ou 1 à distância, enquanto as numéricas
contribuem em escala contínua. Funciona na prática para poucas categorias;
com muitas colunas categóricas de alta cardinalidade, considere distância de
Gower ou codificação diferente antes de confiar no resultado.

In [ ]:
def build_preprocessor(config: DensityClusterConfig) -> ColumnTransformer:
    scaler = StandardScaler() if config.scale_method == "standard" else RobustScaler()

    transformers = []
    if config.numeric_features:
        transformers.append(("num", scaler, config.numeric_features))
    if config.categorical_features:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                              config.categorical_features))

    if not transformers:
        raise ValueError("Configure ao menos uma feature numerica ou categorica.")

    return ColumnTransformer(transformers)

## 4. Estimar `min_samples`

Heurística de Sander et al. (1998) / Schubert et al. (2017, "DBSCAN Revisited"):
`min_samples >= D + 1`; para dados com ruído, `2*D` é ponto de partida mais
seguro (`D` = dimensionalidade **depois** do encoding — uma coluna categórica
vira várias colunas one-hot, e isso conta). Trate como chute inicial: a seção
7 (sensibilidade) mostra se o resultado depende demais dessa escolha.

In [ ]:
def suggest_min_samples(n_features_transformed: int) -> int:
    return max(3, 2 * n_features_transformed)

## 5. Estimar `eps` — k-distance plot + detecção de joelho

Mesma lógica do notebook original (`NearestNeighbors` + `k-distance plot`),
mas agora com a escolha do "joelho" automatizada em vez de decidida no olho:
para cada ponto, calcula a distância ao `min_samples`-ésimo vizinho mais
próximo, ordena crescente. A curva costuma ficar praticamente plana onde os
pontos estão em regiões densas, e sobe de forma abrupta nos pontos
esparsos/ruído — o "joelho" é essa virada. `suggest_eps` acha esse ponto pela
distância máxima à corda que liga o primeiro ao último ponto da curva
(normalizada em [0,1] nos dois eixos, para a geometria não ser enviesada pela
escala) — é a mesma ideia do método kneedle, na versão geométrica simples.

Como no K-Means, trate a sugestão como ponto de partida: releia o gráfico.

In [ ]:
def compute_k_distances(X: np.ndarray, min_samples: int) -> np.ndarray:
    nbrs = NearestNeighbors(n_neighbors=min_samples).fit(X)
    distances, _ = nbrs.kneighbors(X)
    return np.sort(distances[:, -1])


def suggest_eps(k_dists: np.ndarray) -> tuple[float, int]:
    n = len(k_dists)
    x = np.arange(n, dtype=float)
    y = np.asarray(k_dists, dtype=float)

    x_n = (x - x.min()) / (x.max() - x.min() + 1e-12)
    y_n = (y - y.min()) / (y.max() - y.min() + 1e-12)
    x0, y0, x1, y1 = x_n[0], y_n[0], x_n[-1], y_n[-1]

    num = np.abs((y1 - y0) * x_n - (x1 - x0) * y_n + x1 * y0 - y1 * x0)
    den = np.sqrt((y1 - y0) ** 2 + (x1 - x0) ** 2) + 1e-12
    dist_to_chord = num / den

    knee_idx = int(np.argmax(dist_to_chord))
    return float(y[knee_idx]), knee_idx


def plot_k_distance(k_dists: np.ndarray, eps_suggested: Optional[float] = None,
                     min_samples: Optional[int] = None) -> None:
    plt.figure(figsize=(6, 4))
    plt.plot(k_dists)
    if eps_suggested is not None:
        plt.axhline(eps_suggested, color="red", linestyle="--",
                     label=f"eps sugerido = {eps_suggested:.3f}")
        plt.legend()
    plt.title(f"k-distance plot (k={min_samples})")
    plt.xlabel("Pontos ordenados por distancia")
    plt.ylabel(f"Distancia ao {min_samples}-esimo vizinho")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.show()

## 6. Treinar o modelo

In [ ]:
def fit_dbscan(X: np.ndarray, eps: float, min_samples: int, metric: str = "euclidean") -> DBSCAN:
    model = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    model.fit(X)
    return model

## 7. Avaliar — métricas de cluster/ruído + sensibilidade a hiperparâmetro

`evaluate_clustering` reporta nº de clusters, % de ruído e silhouette
calculado **só nos pontos não-ruído** (silhouette não é definido para o rótulo
-1, e incluí-lo distorceria a métrica).

`evaluate_param_sensitivity` é o análogo do teste de estabilidade do K-Means,
mas para outra fonte de variação: como DBSCAN não tem seed, a pergunta não é
"o resultado muda entre reruns?" (não muda) e sim "o resultado muda muito se
eu errar `eps`/`min_samples` por uma margem pequena?". Roda um grid pequeno em
torno dos valores escolhidos (`eps` ±10%, `min_samples` ±1) e mede a
concordância com o resultado base via Adjusted Rand Index. ARI baixo no grid
= escolha de hiperparâmetro frágil, resultado pouco confiável.

In [ ]:
def evaluate_clustering(X: np.ndarray, labels: np.ndarray) -> dict:
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int(np.sum(labels == -1))
    pct_noise = round(n_noise / len(labels) * 100, 1)

    sil = None
    mask = labels != -1
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = float(silhouette_score(X[mask], labels[mask]))

    return {"n_clusters": n_clusters, "n_noise": n_noise, "pct_noise": pct_noise,
            "silhouette_sem_ruido": sil}


def evaluate_param_sensitivity(X: np.ndarray, eps: float, min_samples: int,
                                config: DensityClusterConfig, eps_frac: float = 0.1,
                                ms_delta: int = 1) -> tuple[pd.DataFrame, float]:
    base_labels = DBSCAN(eps=eps, min_samples=min_samples, metric=config.metric).fit_predict(X)

    rows = []
    for eps_mult in (1 - eps_frac, 1.0, 1 + eps_frac):
        for ms_delta_i in (-ms_delta, 0, ms_delta):
            if eps_mult == 1.0 and ms_delta_i == 0:
                continue
            e = eps * eps_mult
            ms = max(2, min_samples + ms_delta_i)
            labels_i = DBSCAN(eps=e, min_samples=ms, metric=config.metric).fit_predict(X)
            rows.append({"eps": round(e, 4), "min_samples": ms,
                         "ari_vs_base": adjusted_rand_score(base_labels, labels_i)})

    grid = pd.DataFrame(rows)
    return grid, float(grid["ari_vs_base"].mean())

## 8. Visualizar

Com 2 features, plota os eixos originais (igual ao notebook original,
inclusive marcando ruído com "x"). Com 3+, projeta em PCA 2D só para
visualização — o modelo continua treinado no espaço original.

In [ ]:
def plot_clusters(X: np.ndarray, labels: np.ndarray, config: DensityClusterConfig) -> None:
    if X.shape[1] == 2:
        coords, xlabel, ylabel = X, "feature 1 (padronizada)", "feature 2 (padronizada)"
    else:
        pca = PCA(n_components=2, random_state=config.random_state)
        coords = pca.fit_transform(X)
        var = pca.explained_variance_ratio_.sum() * 100
        xlabel, ylabel = "PC1", f"PC2  ({var:.1f}% da variancia em 2 componentes)"

    plt.figure(figsize=(6, 5))
    for c in sorted(np.unique(labels)):
        m = labels == c
        if c == -1:
            plt.scatter(coords[m, 0], coords[m, 1], s=50, marker="x", c="black", label="Ruido (-1)")
        else:
            plt.scatter(coords[m, 0], coords[m, 1], s=40, label=f"Cluster {c}")
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    plt.title(f"DBSCAN (clusters={n_clusters}, ruido={int(np.sum(labels == -1))})")
    plt.legend(fontsize=8)
    plt.show()

## 9. Comunicar — perfil por cluster (incluindo ruído)

Ruído vira sua própria linha no perfil (`Ruído (-1)`) — é informação de
negócio tanto quanto qualquer cluster: pode ser cliente atípico, erro de
captura, ou exatamente o caso de fraude/anomalia que você está procurando.

In [ ]:
def profile_clusters(df: pd.DataFrame, labels: np.ndarray, config: DensityClusterConfig) -> pd.DataFrame:
    work = df.copy()
    work["cluster"] = labels

    profile = work.groupby("cluster").agg(n_registros=("cluster", "size"))
    profile["pct_base"] = (profile["n_registros"] / len(work) * 100).round(1)

    for col in config.numeric_features:
        profile[f"{col}_media"] = work.groupby("cluster")[col].mean().round(2)

    for col in config.categorical_features:
        profile[f"{col}_dominante"] = work.groupby("cluster")[col].agg(
            lambda s: s.mode().iat[0] if not s.mode().empty else None
        )

    profile = profile.sort_index()
    profile.index = ["Ruido (-1)" if i == -1 else f"Cluster {i}" for i in profile.index]
    return profile

## 10. Persistir e reaplicar (com ressalva)

`save_pipeline` grava o preprocessor e o `DBSCAN` treinado (que já carrega
`.components_`, as coordenadas dos core points, e `.core_sample_indices_`).

`predict_new` **não é** `.predict()` de verdade — sklearn não oferece isso
para DBSCAN porque o método é transdutivo (o resultado depende do dataset de
treino inteiro; não existe fronteira de decisão fixa aprendida). A aproximação
usada aqui, comum na prática: atribuir o ponto novo ao cluster do core point
mais próximo, se a distância for `<= eps`; caso contrário, tratar como ruído.
Isso pode divergir do que você obteria re-rodando o DBSCAN com o ponto
incluído no treino — reavalie periodicamente com dado real, não trate como
garantia.

In [ ]:
def save_pipeline(preprocessor: ColumnTransformer, model: DBSCAN, config: DensityClusterConfig) -> None:
    with open(config.model_path, "wb") as f:
        pickle.dump({"preprocessor": preprocessor, "model": model, "config": config}, f)


def load_pipeline(model_path: str) -> dict:
    with open(model_path, "rb") as f:
        return pickle.load(f)


def predict_new(df_new: pd.DataFrame, bundle: dict) -> np.ndarray:
    config = bundle["config"]
    model = bundle["model"]
    cols = config.numeric_features + config.categorical_features
    X_new = bundle["preprocessor"].transform(df_new[cols])

    core_points = model.components_
    if len(core_points) == 0:
        return np.full(len(X_new), -1)

    core_labels = model.labels_[model.core_sample_indices_]
    dists = np.linalg.norm(X_new[:, None, :] - core_points[None, :, :], axis=2)
    nearest_idx = np.argmin(dists, axis=1)
    nearest_dist = dists[np.arange(len(X_new)), nearest_idx]
    return np.where(nearest_dist <= model.eps, core_labels[nearest_idx], -1)

## Orquestrador

`eps=None`/`min_samples=None` deixam a sugestão automática decidir (seções 4
e 5). Passe valores explícitos para fixar manualmente. `persist=False` roda o
clustering sem gravar em disco — use isso para comparações/experimentos que
não devem substituir o modelo "oficial" salvo por uma chamada anterior (ver
Exemplo 1 abaixo: comparar `eps` manual com o sugerido não pode sobrescrever
silenciosamente o modelo já salvo).

In [ ]:
def run_density_clustering_pipeline(config: DensityClusterConfig, eps: Optional[float] = None,
                                      min_samples: Optional[int] = None,
                                      show_plots: bool = True, persist: bool = True) -> dict:
    df = load_data(config)
    profile_dataset(df, config)

    preprocessor = build_preprocessor(config)
    cols = config.numeric_features + config.categorical_features
    X = preprocessor.fit_transform(df[cols])

    ms_final = min_samples or config.min_samples or suggest_min_samples(X.shape[1])
    k_dists = compute_k_distances(X, ms_final)
    eps_suggested, _ = suggest_eps(k_dists)
    eps_final = eps or config.eps or eps_suggested

    if show_plots:
        plot_k_distance(k_dists, eps_suggested, ms_final)

    model = fit_dbscan(X, eps_final, ms_final, config.metric)
    labels = model.labels_

    metrics = evaluate_clustering(X, labels)
    sens_grid, sens_ari_mean = evaluate_param_sensitivity(X, eps_final, ms_final, config)

    if show_plots:
        plot_clusters(X, labels, config)

    cluster_profile = profile_clusters(df, labels, config)
    if persist:
        save_pipeline(preprocessor, model, config)

    return {
        "df_labeled": df.assign(cluster=labels),
        "eps_final": eps_final, "min_samples_final": ms_final, "eps_suggested": eps_suggested,
        "metrics": metrics,
        "sensitivity_grid": sens_grid, "sensitivity_ari_mean": sens_ari_mean,
        "cluster_profile": cluster_profile,
        "model": model, "preprocessor": preprocessor,
    }

## Exemplo 1 — cenário original, com sugestão automática

O notebook original fixava `eps=0.5, min_samples=5` escolhidos no olho a
partir do k-distance plot. Aqui deixamos o pipeline sugerir, e comparamos com
o valor original manualmente forçado — pra mostrar que a automação chega perto
do que uma leitura cuidadosa do gráfico dava, e que o override manual continua
disponível.

In [ ]:
result = run_density_clustering_pipeline(CONFIG)
print(f"eps sugerido: {result['eps_suggested']:.3f}  |  eps usado: {result['eps_final']:.3f}"
      f"  |  min_samples usado: {result['min_samples_final']}")
print(f"clusters: {result['metrics']['n_clusters']}  |  ruido: {result['metrics']['pct_noise']}%"
      f"  |  silhouette (sem ruido): {result['metrics']['silhouette_sem_ruido']}")
print(f"sensibilidade a hiperparametro (ARI medio no grid +-10%/+-1): {result['sensitivity_ari_mean']:.3f}")
result["cluster_profile"]

In [ ]:
# Comparação com os valores exatos do notebook original (eps=0.5, min_samples=5).
# persist=False: e' so uma comparacao, nao deve substituir o modelo salvo pelo Exemplo 1 acima.
result_original = run_density_clustering_pipeline(CONFIG, eps=0.5, min_samples=5,
                                                    show_plots=False, persist=False)
print(f"[eps=0.5, min_samples=5 manual] clusters: {result_original['metrics']['n_clusters']}"
      f"  |  ruido: {result_original['metrics']['pct_noise']}%")

## Exemplo 2 — outro cenário, só trocando CONFIG

3 numéricas + 1 categórica (`Genre`), `RobustScaler`. Nenhuma função acima foi
tocada — só o `DensityClusterConfig`. Para RFM/segmentação de funcionários/
detecção de fraude: troque `data_path`, `numeric_features` e
`categorical_features` pelas suas colunas. Se o cenário for RFM de clientes
com foco em segmentação (não em detecção de anomalia), a skill `analise-rfm`
usa K-Means, não DBSCAN — mais adequado quando você quer grupos de tamanho
comparável, não "grupos densos + exceções".

In [ ]:
CONFIG_V2 = DensityClusterConfig(
    data_path="dataset/Mall_Customers.csv",
    id_col="CustomerID",
    numeric_features=["Age", "Annual Income (k$)", "Spending Score (1-100)"],
    categorical_features=["Genre"],
    scale_method="robust",
    model_path="dbscan_pipeline_v2.pkl",
)

result_v2 = run_density_clustering_pipeline(CONFIG_V2)
result_v2["cluster_profile"]

## Aplicando o modelo salvo a um registro novo (aproximação)

O dicionário abaixo é um template: as chaves precisam bater com as colunas do
`CONFIG` usado para treinar. Aqui com as 2 colunas do `CONFIG` original —
atualize se você trocou o `CONFIG` lá em cima.

In [ ]:
bundle = load_pipeline(CONFIG.model_path)
novo_cliente = pd.DataFrame([{"Annual Income (k$)": 80, "Spending Score (1-100)": 75}])
cluster_previsto = predict_new(novo_cliente, bundle)
print("Cluster do novo cliente (-1 = ruido/atipico):", cluster_previsto[0])

## Limitações — vale saber antes de aplicar em qualquer dado

DBSCAN usa um único `eps` global, o que assume densidade razoavelmente
uniforme entre clusters. Se o seu dataset tem um cluster denso e outro
esparso, um `eps` que funciona bem para um funde ou fragmenta o outro — nesse
caso, `HDBSCAN` (hierárquico, sem `eps` fixo) ou `OPTICS` (generaliza DBSCAN
com um gráfico de alcançabilidade, permite ler densidades diferentes num só
resultado) tendem a servir melhor.

Distância euclidiana perde poder discriminante em alta dimensão (curse of
dimensionality) — o mesmo problema do K-Means, mas mais crítico aqui porque
`eps` é um raio fixo: acima de ~10-15 features, considere PCA antes do
clustering (não só para visualizar, como na seção 8, mas como input real do
DBSCAN). É sensível a escala (por isso o `ColumnTransformer` da seção 3) e a
`eps`/`min_samples` (por isso a seção 7 de sensibilidade) — ao contrário do
K-Means, aqui não existe "silhouette decide sozinho": os dois hiperparâmetros
interagem e precisam de leitura humana do k-distance plot.

## Extra didático (opcional) — animações de varredura de eps e min_samples

Isoladas do pipeline principal de propósito: mostram visualmente como o
resultado muda ao variar um hiperparâmetro de cada vez — é o mesmo papel que
a seção 7 (sensibilidade) cumpre numericamente. Requer `imageio`. Funcionam
só com exatamente 2 features (plotam eixos originais, sem PCA, igual ao
notebook original) — não são chamadas por `run_density_clustering_pipeline`.

In [ ]:
def animate_eps_sweep(X_raw: np.ndarray, X_scaled: np.ndarray, eps_values: list, min_samples: int,
                       gif_path: str = "dbscan_eps.gif", folder: str = "dbscan_eps_frames"):
    import os
    import imageio
    from IPython.display import Image

    os.makedirs(folder, exist_ok=True)
    frames = []
    for e in eps_values:
        labels_tmp = DBSCAN(eps=e, min_samples=min_samples).fit_predict(X_scaled)
        uniq = np.unique(labels_tmp)
        n_clusters = len(set(labels_tmp)) - (1 if -1 in labels_tmp else 0)
        n_noise = int(np.sum(labels_tmp == -1))

        plt.figure(figsize=(6, 5))
        for c in uniq:
            m = labels_tmp == c
            if c == -1:
                plt.scatter(X_raw[m, 0], X_raw[m, 1], s=50, marker="x", label="Ruido (-1)")
            else:
                plt.scatter(X_raw[m, 0], X_raw[m, 1], s=50, label=f"Cluster {c}")
        plt.title(f"DBSCAN eps={e}, min_samples={min_samples}\nclusters={n_clusters}, ruido={n_noise}")
        plt.legend(loc="best", fontsize=8)

        fname = f"{folder}/eps_{str(e).replace('.', '_')}.png"
        plt.savefig(fname, dpi=140, bbox_inches="tight")
        plt.close()
        frames.append(fname)

    with imageio.get_writer(gif_path, mode="I", duration=3000, loop=0) as writer:
        for f in frames:
            writer.append_data(imageio.v2.imread(f))
    return Image(filename=gif_path)


def animate_min_samples_sweep(X_raw: np.ndarray, X_scaled: np.ndarray, ms_values: list, eps: float,
                               gif_path: str = "dbscan_min_samples.gif",
                               folder: str = "dbscan_min_samples_frames"):
    import os
    import imageio
    from IPython.display import Image

    os.makedirs(folder, exist_ok=True)
    frames = []
    for ms in ms_values:
        labels_tmp = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_scaled)
        uniq = np.unique(labels_tmp)
        n_clusters = len(set(labels_tmp)) - (1 if -1 in labels_tmp else 0)
        n_noise = int(np.sum(labels_tmp == -1))

        plt.figure(figsize=(6, 5))
        for c in uniq:
            m = labels_tmp == c
            if c == -1:
                plt.scatter(X_raw[m, 0], X_raw[m, 1], s=50, marker="x", label="Ruido (-1)")
            else:
                plt.scatter(X_raw[m, 0], X_raw[m, 1], s=50, label=f"Cluster {c}")
        plt.title(f"DBSCAN eps={eps}, min_samples={ms}\nclusters={n_clusters}, ruido={n_noise}")
        plt.legend(loc="best", fontsize=8)

        fname = f"{folder}/ms_{ms}.png"
        plt.savefig(fname, dpi=140, bbox_inches="tight")
        plt.close()
        frames.append(fname)

    with imageio.get_writer(gif_path, mode="I", duration=3000, loop=0) as writer:
        for f in frames:
            writer.append_data(imageio.v2.imread(f))
    return Image(filename=gif_path)

# animate_eps_sweep(X, X_scaled, [.1,.3,.35,.4,.45,.5,.55,.6,.8], min_samples=5)  # descomente p/ gerar
# animate_min_samples_sweep(X, X_scaled, [3,4,5,6,8,10,12,15], eps=.35)          # descomente p/ gerar